In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightcurvelynx.obstable.ztf_obstable import ZTFObsTable, _ztfcam_ccd_gain
from lightcurvelynx.astro_utils.mag_flux import mag2flux
import sqlite3

In [2]:
con = sqlite3.connect("data/ztf_metadata_latest.db")
sql_query = "SELECT * FROM exposures"
metadata_table = pd.read_sql_query(sql_query, con)
metadata_table = metadata_table.replace("", np.nan)
metadata_table = metadata_table.dropna(subset=["fwhm"])
metadata_table.columns

/var/folders/zs/zxl3t6ks12zg2l3dp9qn1rkr0000gn/T/ipykernel_73708/2534340399.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_table = metadata_table.replace("", np.nan)


Index(['expid', 'field', 'filter', 'obsdate', 'ra', 'dec', 'exptime',
       'airmass', 'infobits', 'dr', 'numsci', 'numdiff', 'fwhm', 'maglim',
       'scibckgnd', 'ellip', 'ellippa'],
      dtype='object')

In [3]:
obs_log = pd.read_parquet('ztfsniadr2/tables/observing_logs.parquet')
obs_log = obs_log.drop_duplicates("expid")
obs_log["filter"] = obs_log.apply(lambda row: row["band"][-1],axis=1)
obs_log = pd.merge(obs_log, metadata_table[["expid","filter","exptime","fwhm","obsdate","scibckgnd","ra","dec"]],on=["filter","expid"])
gain = _ztfcam_ccd_gain
obs_log["zp_nJy"] = mag2flux(obs_log["zp"].values + 2.5*np.log10(gain))
obs_log = obs_log.rename(columns={"zp":"zp_abmag"})

In [4]:
obs_log

,mjd,band,fieldid,fieldra,fielddec,rcid,maglimit,zp_abmag,gain,expid,infobits,skynoise,filter,exptime,fwhm,obsdate,scibckgnd,ra,dec,zp_nJy
0,58288.171875,ztfi,375,3.787257,-0.171915,16,19.650887,25.644886,6.2,53417042,0,49.960846,i,30.0,3.467395,2018-06-19 04:05:25.548,334.3445,216.993833,-9.85,32.333588
1,58288.171875,ztfi,427,3.850371,-0.046251,16,19.689102,25.650101,6.2,53417087,0,48.465134,i,30.0,3.543785,2018-06-19 04:06:04.871,297.5000,220.610000,-2.65,32.178665
2,58288.171875,ztfi,478,3.833042,0.079412,17,19.878822,25.654823,6.2,53417134,0,40.872425,i,30.0,2.973075,2018-06-19 04:06:44.296,329.7940,219.617125,4.55,32.039003
3,58288.171875,ztfi,529,3.754645,0.205076,16,20.252375,25.659374,6.2,53417179,0,29.095768,i,30.0,2.407530,2018-06-19 04:07:23.921,234.9170,215.125333,11.75,31.904990
4,58288.171875,ztfi,580,3.774032,0.330740,17,20.534861,25.681860,6.2,53417225,0,22.899696,i,30.0,2.076620,2018-06-19 04:08:03.345,232.5710,216.236083,18.95,31.251029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522192,59273.554688,ztfr,490,5.273359,0.079412,22,18.371059,26.197060,6.3,151955548,0,270.041229,r,30.0,4.445800,2021-02-28 13:19:54.776,142.7340,302.141208,4.55,19.443948
522193,59273.554688,ztfr,438,5.208885,-0.046251,18,18.128563,26.095562,6.4,151955596,0,307.489014,r,30.0,4.250080,2021-02-28 13:20:35.531,178.6410,298.447125,-2.65,21.349294
522194,59273.554688,ztfr,642,5.511889,0.456404,17,18.066133,26.078133,6.5,151955651,0,320.501190,r,30.0,4.535925,2021-02-28 13:21:23.577,129.9455,315.808000,26.15,21.694780
522195,59273.558594,ztfr,593,5.421456,0.330740,22,17.839561,26.214561,6.3,151955697,33554440,447.744232,r,30.0,4.731005,2021-02-28 13:22:04.025,132.0290,310.626542,18.95,19.133029


In [5]:
# remove wrong ra/dec values 
print("before:",len(obs_log))
idx = obs_log.ra<0
display(obs_log.loc[idx])
obs_log = obs_log.loc[~idx]
print("after",len(obs_log))

before: 522197


,mjd,band,fieldid,fieldra,fielddec,rcid,maglimit,zp_abmag,gain,expid,infobits,skynoise,filter,exptime,fwhm,obsdate,scibckgnd,ra,dec,zp_nJy
112646,58493.554688,ztfg,1620,3.100007,0.393572,17,22.095169,26.252171,6.2,73955409,0,9.200906,g,30.0,1.933235,2019-01-10 13:17:54.175,154.36300,-149985.0,-9999.0,18.481622
140094,58577.445312,ztfr,588,4.788912,0.330740,17,16.654907,21.897907,6.2,82344436,48,25.016699,r,30.0,1.814035,2019-04-04 10:39:53.285,3.76500,-149985.0,-9999.0,1019.636580
197317,58698.285156,ztfr,865,4.817109,1.336050,21,19.936420,25.035919,6.2,94428439,0,21.919441,r,30.0,1.784935,2019-08-03 06:49:32.285,72.45985,-149985.0,-9999.0,56.655312
226618,58743.390625,ztfg,852,5.834386,1.210386,17,19.818365,26.074366,6.2,98939162,0,63.596058,g,30.0,2.756865,2019-09-17 09:23:56.285,91.55705,-149985.0,-9999.0,21.770182
270140,58799.511719,ztfg,701,1.100809,0.707731,18,19.510597,26.122597,6.4,104551297,0,88.273392,g,30.0,2.096860,2019-11-12 12:18:41.764,90.42550,-149985.0,-9999.0,20.824262


after 522192


In [6]:
obs_log.infobits.unique()

array([       0,        8,  2097152,     2056,        1, 33554440,
           2072,     2048,       48,     2096,       32,     2080,
           2088,        9,       49,     2049,       51,       24,
             40,     2104,       56,     2097,     2065,     2099,
           2083,     2050,        2,        3,       19,       57,
             17,       59,     2081,       33,     2051,     2067,
             16,       35, 33554480, 33554432, 33554464, 33556480,
       33556512, 33556528,       41,       25,  4194304,      513,
       37748736, 35651584,  2097153, 33554481,     2073,     2075,
        2097171,       50,  2099202,  2099200,      512,  4194312,
        4194336,  4194352,  4194305,  4194353,  4194816,      520,
        4196352,  6291456,  4194313, 37750840, 33554433, 33554944,
       33554456, 33554488, 33554457, 37748744, 33554489, 37748785,
       37748739, 33556513, 33556529, 33554435, 33554483, 37750835,
       33556531, 33554448, 33554441, 33556488], dtype=uint32)

In [7]:
# remove bad quality epochs
# page 4 of https://irsa.ipac.caltech.edu/data/ZTF/docs/ztf_extended_cautionary_notes.
# " If INFOBITS for an image has value < 33554432 (i.e., does not contain bit 25), the image and catalog data are probably usable. "

print("before:", len(obs_log))
obs_log = obs_log.loc[obs_log.infobits < 33554432]
print("after:", len(obs_log))

before: 522192
after: 520115


In [8]:
obs_log.to_parquet("data/ztf_observing_log_combined_w_metadata.parquet")

In [9]:
# from astropy import units as u
# from astropy.coordinates import SkyCoord
# c = SkyCoord(ra=obs_log.fieldra.values*u.radian, dec=obs_log.fielddec.values*u.radian, frame='icrs')
# obs_log['fieldra_deg'] = c.ra.degree
# obs_log['fileddec_deg'] = c.dec.degree
# print(np.sum(np.isclose(c.ra.degree,obs_log.ra,atol=0.005)))
# print(np.sum(np.isclose(c.dec.degree,obs_log.dec,atol=0.005)))
# print(len(obs_log))

In [10]:
# idx = np.isclose(c.ra.degree,obs_log.ra,atol=0.005)
# obs_log.loc[~idx]